# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [2]:
!pip install -qU "langchain>=1.0" "langchain-core>=0.3" "langchain-community>=0.4" "langchain-classic>=0.3" langchain-openai langchain-elasticsearch tiktoken jq lark elasticsearch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.8/773.8 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.8/952.8 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.3/235.3 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0

In [3]:
import os
from getpass import getpass
from langchain_openai.embeddings import OpenAIEmbeddings
#from langchain_elasticsearch import ElasticsearchStore
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever  # ← CORRECT import for 1.0+


from langchain_community.vectorstores.elasticsearch import ElasticsearchStore
#from langchain_openai import OpenAIEmbeddings

# os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [4]:
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial.

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [7]:
0# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
ELASTIC_CLOUD_ID = userdata.get('ELASTIC_CLOUD_ID')

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
ELASTIC_API_KEY = userdata.get('ELASTIC_API_KEY')

# https://platform.openai.com/api-keys
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')


# Create OpenAI embedding model
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

# Set a meaningful index name
INDEX_NAME = 'iron_chat_multiquery' #give it a meaningful name,

# Create Elasticsearch vector store
vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name=INDEX_NAME,
    embedding=embeddings  # ✅ Add this line
)


## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [8]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [9]:
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json  # Optional: for metadata extraction
from datetime import datetime # Import datetime here


def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    None
    return metadata



loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",  # Extracts array of records
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000,       # e.g., ~750 words
    chunk_overlap=200,     # Overlap for context preservation
)

docs = loader.load_and_split(text_splitter=text_splitter)


### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [10]:
from datetime import datetime
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

# Clean docs metadata
clean_docs = []
for doc in docs:
    metadata = doc.metadata.copy()

    if metadata.get("updated_at") in ["", None, "null"]:
        metadata["updated_at"] = datetime.now().isoformat()

    # Use model_copy() instead of copy() to avoid Pydantic deprecation warning
    clean_docs.append(doc.model_copy(update={"metadata": metadata}))

# Create embeddings ONCE
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

# Create vectorstore ONCE using cleaned docs
vectorstore = ElasticsearchStore.from_documents(
    clean_docs,
    embeddings,
    index_name=INDEX_NAME,
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

# Create LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    openai_api_key=OPENAI_API_KEY # Explicitly pass the API key
)

# Create MultiQueryRetriever
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    llm=llm
)


# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [11]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import chain as lc_chain
import logging

# Enable detailed logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Multi-query generator with 3 variants
MULTI_QUERY_PROMPT = ChatPromptTemplate.from_template("""
Generate 3 diverse versions of this question for better retrieval. Vary phrasing, keywords, and perspectives:

{question}

Queries (one per line):
""")

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template("""
📄 [{source}]
{page_content}
---
""")

# Define the context prompt for the LLM
LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}

Question: {question}
""")

def safe_combine_docs(docs):
    """Production-ready doc formatting with fallbacks"""
    doc_strings = []
    for i, doc in enumerate(docs):
        try:
            doc_dict = doc.model_dump()
            source = doc.metadata.get("name") or doc.metadata.get("source", f"Doc-{i}")
            doc_dict["source"] = source
            formatted = LLM_DOCUMENT_PROMPT.format(**doc_dict)
        except Exception as e:
            logger.warning(f"Doc format error: {e}")
            formatted = f"[Doc-{i}] {doc.page_content[:500]}..."
        doc_strings.append(formatted)
    return "\n\n".join(doc_strings)

# Self-healing chain: retry bad retrievals
def self_healing_retriever(query, max_tries=2):
    """Retry with rewritten query if empty results"""
    for attempt in range(max_tries):
        docs = retriever.invoke(query)
        if docs:
            return docs
        logger.info(f"Empty results (attempt {attempt+1}), rewriting...")
        query = llm.invoke(f"Rewrite for better retrieval: {query}").content
    return retriever.invoke(query)  # Fallback

_context = RunnableParallel(
    context=(RunnablePassthrough() | self_healing_retriever | safe_combine_docs),
    question=RunnablePassthrough(),
)

rag_chain = _context | LLM_CONTEXT_PROMPT | llm | StrOutputParser()

# Test with auto-multi-query
def multi_query_rag(question):
    """Generate + retrieve + answer"""

    query_chain = MULTI_QUERY_PROMPT | llm | StrOutputParser()

    generated_queries = query_chain.invoke({"question": question})

    print("\nGenerated Queries:")
    print("------------------")
    print(generated_queries)
    print("------------------\n")

    queries = [q.strip() for q in generated_queries.split("\n") if q.strip()]

    all_docs = []

    for q in queries:
        docs = self_healing_retriever(q)
        all_docs.extend(docs[:3])

    return rag_chain.invoke({
        "question": question,
        "context": safe_combine_docs(all_docs)
    })

print("---- Answer ----")

print(multi_query_rag("what is the nasa sales team?"))

---- Answer ----

Generated Queries:
------------------
1. Can you provide information on the sales team at NASA?
2. What is the role of the sales team within NASA?
3. How does NASA's sales team operate and contribute to the organization's goals?
------------------

The NASA sales team consists of two Area Vice-Presidents: Laura Martinez for North America and Gary Johnson for South America.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

In [12]:
import json
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Create a highly specific system prompt for structured query expansion
CREATIVE_JSON_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You are an expert AI research assistant. Your task is to analyze the user's question "
               "and break it down into 3 distinct, highly targeted search queries to fetch information "
               "from a vector database. Provide your response ONLY as a raw valid JSON list of strings."),
    ("user", "Generate 3 diverse search queries for: {question}")
])

def custom_json_multi_query_rag(question: str):
    print(f"=== Iteration 1: JSON Structured Query Expansion ===")

    # Generate queries as a JSON string
    json_chain = CREATIVE_JSON_PROMPT | llm | StrOutputParser()
    raw_json = json_chain.invoke({"question": question})

    try:
        queries = json.loads(raw_json)
    except Exception:
        # Fallback if JSON parsing fails
        queries = [question]

    print("Generated Target Queries:")
    for idx, q in enumerate(queries, 1):
        print(f"  {idx}. {q}")
    print("-" * 40)

    # Collect and gather document contexts
    all_docs = []
    for q in queries:
        docs = self_healing_retriever(q)
        all_docs.extend(docs[:2]) # Grab top 2 from each query to maximize diversity

    # Generate final answer using your existing RAG chain setup
    answer = rag_chain.invoke({
        "question": question,
        "context": safe_combine_docs(all_docs)
    })
    return answer

# Test Iteration 1
print(custom_json_multi_query_rag("What are the corporate policies on remote work and flexible hours?"))

=== Iteration 1: JSON Structured Query Expansion ===
Generated Target Queries:
  1. Corporate policies remote work
  2. Corporate policies flexible hours
  3. Remote work guidelines corporations
----------------------------------------
The corporate policies on remote work and flexible hours include guidelines for employees to work from home full-time, maintain the same level of performance and collaboration as in the office, communicate effectively, maintain regular work hours, track work hours accurately, prioritize health and well-being, and adhere to confidentiality and data security policies. Employees must also seek approval for overtime, maintain a safe and productive workspace, and address any questions or concerns to their supervisor or the HR department.


In [13]:
# 1. Prompt the LLM to split the question into atomic, complementary sub-components
DECOMPOSITION_PROMPT = ChatPromptTemplate.from_template("""
Deconstruct the following complex user inquiry into exactly 3 smaller, distinct sub-questions
needed to compile a complete answer. Do not repeat the same question.

Complex Question: {question}

Sub-questions (one per line):
""")

def sub_question_decomposition_rag(question: str):
    print(f"=== Iteration 2: Sub-Question Decomposition ===")

    decomp_chain = DECOMPOSITION_PROMPT | llm | StrOutputParser()
    raw_sub_questions = decomp_chain.invoke({"question": question})

    sub_questions = [q.strip() for q in raw_sub_questions.split("\n") if q.strip()]

    print("Decomposed Sub-Questions:")
    for idx, sq in enumerate(sub_questions, 1):
        print(f"  {idx}. {sq}")
    print("-" * 40)

    # Fetch specialized context for each sub-question
    all_docs = []
    for sq in sub_questions:
        docs = self_healing_retriever(sq)
        all_docs.extend(docs[:2])

    # Execute the final response generation
    answer = rag_chain.invoke({
        "question": question,
        "context": safe_combine_docs(all_docs)
    })
    return answer

# Test Iteration 2
print(sub_question_decomposition_rag("Who runs the sales operations in North America and how can I contact them?"))

=== Iteration 2: Sub-Question Decomposition ===
Decomposed Sub-Questions:
  1. 1. Who is in charge of sales operations in North America?
  2. 2. What is the contact information for the person in charge of sales operations in North America?
  3. 3. How can I reach out to the person responsible for sales operations in North America?
----------------------------------------
The sales operations in North America are run by Laura Martinez, the Area Vice-President of North America. You can contact her through the sales organization structure outlined in the provided context.
